In [ ]:
from pathlib import Path
import rasterio
import geopandas as gpd
import numpy as np
from rasterstats import zonal_stats
from shapely.geometry import box
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# -----------------------------
# 1. OPEN RASTER S2
# -----------------------------
raster_path = Path(r"_data_\Parika\s2")
band_s2 = "RedEdge_res.tif"
r = rasterio.open(raster_path / band_s2)

# Get raster bounds + CRS
bounds = r.bounds
crs = r.crs
res_x, res_y = r.res  # resolution (pixel size)

print("Raster resolution:", res_x, res_y)
print("Raster bounds:", bounds)
print("Raster CRS:", crs)

In [ ]:
# -----------------------------
# 2. CREATE GRID BASED ON RESOLUTION
# -----------------------------
xmin, ymin, xmax, ymax = bounds

# Coordinate arrays at pixel size
xs = np.arange(xmin, xmax, res_x)
ys = np.arange(ymin, ymax, res_y)

polygons = []
ids = []

id_counter = 0
for y in ys:
    for x in xs:
        poly = box(x, y, x + res_x, y + res_y)
        polygons.append(poly)
        ids.append(id_counter)
        id_counter += 1

# -----------------------------
# 3. TO GEOPANDAS
# -----------------------------
gdf = gpd.GeoDataFrame({"id": ids}, geometry=polygons, crs=crs)

# -----------------------------
# 4. SAVE GRID
# -----------------------------
output = r"sentinel2_grid_pixels.geojson"
gdf.to_file(output, driver="GeoJSON")

print("Grid saved at:", output)

In [ ]:
grid_pol = r"sentinel2_grid_pixels.geojson"
study_area = r"boundaries.geojson"

In [ ]:
grid = gpd.read_file(grid_pol)
study_area = gpd.read_file(study_area)

In [ ]:

# ----------------------------------------------------------
# Reproject grid if needed (CRS must match)
# ----------------------------------------------------------
if grid.crs != study_area.crs:
    study_area = study_area.to_crs(grid.crs)

# ----------------------------------------------------------
# Spatial filter to speed up
#    (keeps only polygons whose bounding box intersects)
# ----------------------------------------------------------
grid = grid[grid.intersects(study_area.unary_union)]

# ----------------------------------------------------------
# Keep only polygons *completely inside* study area
#    using .within() (strict containment)
# ----------------------------------------------------------
inside = grid[grid.within(study_area.unary_union)]

# ----------------------------------------------------------
# Save the clipped grid
# ----------------------------------------------------------
inside.to_file(r"sentinel2_grid_pixels_clip.geojson", driver = "GeoJSON")

print(f"Original grid cells: {len(grid)}")
print(f"Cells completely inside study area: {len(inside)}")

In [ ]:
band = "red_edge"

In [ ]:
grid = gpd.read_file(r"sentinel2_grid_pixels_clip.geojson")

In [ ]:
# ----------------------------------------------------------
# Open high resolution raster grid (UAV)
# Extract statistics of pixels to the grids in polygon format
# ----------------------------------------------------------

highres_raster_path = Path(r"parrot")
name_raster_high = f"band_{band}.tif"

highres_stats = zonal_stats(
    vectors=grid,
    raster=highres_raster_path / name_raster_high,
    stats="mean",
    nodata=-9999,
    geojson_out=False
)
print("extracted statsitics from UAV")
grid[f"{band}_hr"] = [x["mean"] for x in highres_stats]


In [ ]:
s2_raster = raster_path /  band_s2
print(s2_raster)

In [ ]:
## Extract the values from Sentinel-2 to the polygon grid

s2_stats = zonal_stats(
    vectors=grid,
    raster=raster_path / band_s2,
    stats="mean",
    nodata=-9999,
    geojson_out=False
)
print(s2_stats)
grid[f"{band}_s2"] = [x["mean"] for x in s2_stats]


df = grid[[f"{band}_hr", f"{band}_s2"]].dropna()


In [ ]:
# Sentinel-2 values to float32 reflectance to be comparable. 
df[f"{band}_s2_rs"] = df[f"{band}_s2"] / 10000

In [ ]:
# remove invalid pixels from UAV images.
df_values = df.loc[df[f"{band}_hr"] > 0]

In [ ]:

# ----------------------------------------------------------
# Compute correlation
# ----------------------------------------------------------
r, p = stats.pearsonr(df_values[f"{band}_hr"], df_values[f"{band}_s2_rs"])

print(f"Pearson r = {r:.3f}")
print(f"P-value   = {p:.3e}")


In [ ]:
sns.set_theme('notebook')

In [ ]:
#PLOT THE CORRELATIONS, BAND BY BAND.
plt.figure(figsize=(16, 16))
sns.regplot( x=df_values[f"{band}_hr"], y=df_values[f"{band}_s2_rs"], marker = ".",
                          color="grey",#'#680101',
                          scatter_kws={'alpha': 0.4, 
                                       's': 142}, 
                          line_kws={'color': 'black', 'lw': 3, 'alpha': 0.8 },order = 1) #'linestyle':"--"
#plt.annotate(f"Pearson r = {r:.2f}", 
        #     xy=(0.1, 0.80), xycoords='axes fraction', fontsize=50, weight = "bold") 
plt.axline([0, 0], [1, 1], linestyle = "--", lw = 3, color = "#1F1E1E")
#plt.ylabel("Sentinel-2 NDVI")
#plt.xlabel("High-Resolution NDVI")
plt.ylabel("")
plt.xlabel("")
plt.xticks(fontsize = 24)
plt.yticks(fontsize = 24)
plt.title(f"Correlation {band.upper()} band", fontsize=18, fontweight="bold")
plt.annotate(f"r = {round(r,3)}", 
            xy=(0.05, 0.80), xycoords='axes fraction', fontsize=24, weight = "bold") 
plt.ylim(0, 1.0)
plt.xlim(0, 1.0)

        